# Concrete Mix Optimization: Genetic Algorithm Exploration

This notebook demonstrates the Genetic Algorithm (GA) optimization process step-by-step. We optimize for high strength while minimizing carbon and considering curing time.

In [ ]:
%matplotlib inline
from src.ga import GeneticOptimizer
from src.models import StrengthPredictor
from src.chemistry_simple import calculate_embodied_carbon, estimate_curing_time
from src.chemistry_advanced import analyze_mix, inverse_plan_mix
from src.viz import plot_ga_progress

# Initialize Predictor
predictor = StrengthPredictor()
param_names = ["cement", "slag", "ash", "water", "superplasticizer", "coarse_agg", "fine_agg", "age"]

## 1. Define Multi-Objective Fitness
We want to maximize:
$$ Fitness = Strength - (Carbon \times Weight) - (Time \times Weight) $$

In [ ]:
def multi_objective_fitness(x):
    strength = predictor.predict(x)
    mix_dict = {name: val for name, val in zip(param_names, x)}
    carbon = calculate_embodied_carbon(mix_dict)
    days = estimate_curing_time(mix_dict)
    score = strength - (carbon * 0.1) - (days * 1.0)
    return score

bounds = [
    (100, 550), (0, 360), (0, 200), (120, 250), (0, 30), (700, 1150), (550, 1000), (1, 365)
]

## 2. Run Optimization
Perform the GA search and track generation progress.

In [ ]:
optimizer = GeneticOptimizer(multi_objective_fitness, bounds, pop_size=100, mutation_rate=0.05)

for g in range(50):
    stats = optimizer.step()
    if (g + 1) % 10 == 0:
        print(f"Gen {g+1}: Best Score = {stats['best_fitness']:.2f}, Avg = {stats['avg_fitness']:.2f}")

plot_ga_progress(optimizer.history)

## 3. Advanced Molecular Analysis
Using the chemistry_advanced module for Bogue calculations and hydration kinetics.

**Status:** this layer is dimensionally checked (units are consistent kg/m3 throughout, the
portlandite budget is conserved across SCMs) but **UNCALIBRATED** -- there is no calorimetry
or degree-of-hydration data in this repo to validate it against. Contrast with the carbon
figures reported here, which come from `carbon_from_clinker` -- measured and tested.

In [ ]:
best_mix, best_score = optimizer.get_best()
mix_dict = {name: val for name, val in zip(param_names, best_mix)}

report = analyze_mix(mix_dict)

print("=== Molecular Analysis of Best Candidate ===")
print("NOTE: this hydration/Bogue layer is dimensionally checked but UNCALIBRATED --")
print("no calorimetry or degree-of-hydration data backs it. Contrast with the carbon")
print("figure below, which IS measured and tested (see chemistry_advanced.py docstring).")
if report["clinker_phases"] is not None:
    print(f"Clinker Phases: C3S={report['clinker_phases'].C3S:.1f}%, C2S={report['clinker_phases'].C2S:.1f}%")
    print(f"Degree of Hydration: {report['hydration'].degree_of_hydration:.2%}")
    print(f"Heat released: {report['hydration'].heat_kJ_per_kg_cement:.1f} kJ/kg cement")
else:
    print(f"Clinker phases unavailable: {report['phases_unavailable_reason']}")
print(f"Total C-S-H (incl. pozzolanic/latent-hydraulic): {report['total_CSH_kg_m3']:.1f} kg/m3")
print(f"Clinker-based CO2: {report['carbon_kg_m3']:.1f} kg/m\u00b3")

## 4. Inverse Design (Generative Mode)
Given target properties, generate a plausible mix design.

In [ ]:
target_strength = 45  # MPa
target_carbon = 250   # kg CO2/m³
max_cost = 100        # $/m³

generated_mix = inverse_plan_mix(target_strength, target_carbon, max_cost)
print("Generated Mix for Target Properties:")
for k, v in generated_mix.items():
    print(f"  {k}: {v:.1f}")